In [1]:
import requests
import json
import re
from datetime import datetime

In [2]:
# Define the API endpoint
url = "https://plan.navcanada.ca/weather/api/alpha/?site=CZYZ&alpha=notam&notam_choice=default"

# Fetch the data
response = requests.get(url)
data = response.json()

In [4]:
notams_list = []

for item in data.get("data", []):
    raw_text = json.loads(item.get("text", "{}")).get("raw", "")
    if raw_text:
        start_time = item.get("startValidity")
        end_time = item.get("endValidity")
        
        # Try to extract coordinates (if any)
        matches = extract_dms(raw_text)
        coords = []
        for match in matches:
            lat_dd, lon_dd = dms_to_dd(match[0], match[1])
            coords.append({"latitude": lat_dd, "longitude": lon_dd})
        
        notams_list.append({
            "raw": raw_text,
            "startValidity": start_time,
            "endValidity": end_time,
            "coordinates": coords
        })

In [ ]:
# Filter step
keywords = ["PARAJUMP", "PARACHUTE", "ADVISORY AREA", "CRANE", "GPS", "RESTRICTED"]

filtered_notams = [
    notam for notam in notams_list
    if any(keyword in notam["raw"].upper() for keyword in keywords)
]

# # Save to JSON
# output_filename = "filtered_notams.json"
# with open(output_filename, "w") as f:
#     json.dump(filtered_notams, f, indent=2)

# print(f"Saved {len(filtered_notams)} filtered NOTAMs to {output_filename}")

Saved 382 NOTAMs to notams_parsed.json


In [14]:
## pring the output
## print(json.dumps(notams_list[:2], indent=2))

Saved 47 filtered NOTAMs to filtered_notams.json


## get the DD from the filtered NOTAMS list

In [9]:
import re

def extract_dms_coords(text):
    # Regex to find pairs of Lat/Lon in DMS
    pattern = r'(\d{6}[NS])\s*(\d{7}[EW])'
    matches = re.findall(pattern, text)
    return matches

In [10]:
def dms_to_dd(dms_lat, dms_lon):
    # Latitude
    lat_deg = int(dms_lat[0:2])
    lat_min = int(dms_lat[2:4])
    lat_sec = int(dms_lat[4:6])
    lat_dir = dms_lat[-1]
    lat_dd = lat_deg + lat_min / 60 + lat_sec / 3600
    if lat_dir == 'S':
        lat_dd = -lat_dd

    # Longitude
    lon_deg = int(dms_lon[0:3])
    lon_min = int(dms_lon[3:5])
    lon_sec = int(dms_lon[5:7])
    lon_dir = dms_lon[-1]
    lon_dd = lon_deg + lon_min / 60 + lon_sec / 3600
    if lon_dir == 'W':
        lon_dd = -lon_dd

    return round(lat_dd, 6), round(lon_dd, 6)
    

In [11]:
filtered_notams = []

for notam in notams_list:
    raw_text = notam["raw"].upper()  # Uppercase for case-insensitive matching
    if any(keyword in raw_text for keyword in keywords):
        # Extract DMS coordinates
        matches = extract_dms_coords(raw_text)
        dd_coords = []
        for lat, lon in matches:
            lat_dd, lon_dd = dms_to_dd(lat, lon)
            dd_coords.append(f"{lat_dd:.6f}, {lon_dd:.6f}")
        
        # Add coordinates to NOTAM entry
        notam["coordinates_dd"] = dd_coords
        
        filtered_notams.append(notam)


In [12]:
output_filename = "filtered_notams_with_coords.json"
with open(output_filename, "w") as f:
    json.dump(filtered_notams, f, indent=2)

print(f"Saved {len(filtered_notams)} filtered NOTAMs with coordinates to {output_filename}")


Saved 47 filtered NOTAMs with coordinates to filtered_notams_with_coords.json
